In [ ]:
# Group-30 CodeGen — Application UI
# Two Qwen2.5-Coder-7B LoRA adapters (NL->Python, Python->C++) on one shared base.
# Use a GPU runtime (Runtime -> Change runtime type -> T4/L4 GPU).
!pip -q install gradio "peft>=0.11" "transformers>=4.44" accelerate bitsandbytes

In [ ]:
import os, glob, zipfile, subprocess, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
REPO   = "nayanjha16/CodeGen-Implementations-May_26"
BRANCH = "Group-30-model"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Find the repo's model dir no matter where it was cloned
_hits = glob.glob("/content/**/Group-30/src/model", recursive=True) + glob.glob("Group-30/src/model")
MODEL_DIR = _hits[0] if _hits else "Group-30/src/model"
print("Model dir:", MODEL_DIR)

ADAPTERS = {
    "NL -> Python":  "Qwen_NL_to_PL_LORA_Adapter",
    "Python -> C++": "Qwen_Python_to_CPP_LORA_Adapter",
}
WORK = "/content/adapters"; os.makedirs(WORK, exist_ok=True)

In [ ]:
# Ensure each adapter zip is a REAL file (not a 130-byte Git-LFS pointer), then unzip.
def ensure_adapter(folder):
    zip_path = os.path.join(MODEL_DIR, folder + ".zip")
    out_dir  = os.path.join(WORK, folder)
    if os.path.exists(os.path.join(out_dir, "adapter_config.json")):
        print("Already unzipped:", out_dir); return out_dir
    if (not os.path.exists(zip_path)) or os.path.getsize(zip_path) < 100_000:
        url = f"https://media.githubusercontent.com/media/{REPO}/{BRANCH}/Group-30/src/model/{folder}.zip"
        print("Fetching real zip via LFS media URL...")
        subprocess.run(["curl", "-sL", url, "-o", zip_path], check=True)
    print("Unzipping %s (%.0f MB)" % (folder, os.path.getsize(zip_path)/1e6))
    with zipfile.ZipFile(zip_path) as z: z.extractall(WORK)
    return out_dir

ADAPTER_PATHS = {task: ensure_adapter(folder) for task, folder in ADAPTERS.items()}
print(ADAPTER_PATHS)

In [ ]:
# Load the base model ONCE (4-bit, fits a T4/L4), then attach BOTH adapters.
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token_id is None: tok.pad_token_id = tok.eos_token_id

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                          bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, device_map="auto",
                                            quantization_config=bnb, torch_dtype=torch.bfloat16)

TASKS = list(ADAPTERS.keys()); first, *rest = TASKS
model = PeftModel.from_pretrained(base, ADAPTER_PATHS[first], adapter_name=first)
for t in rest: model.load_adapter(ADAPTER_PATHS[t], adapter_name=t)
model.eval()
print("Loaded base + adapters:", TASKS)

In [ ]:
def build_prompt(task, text):
    if task == "Python -> C++":
        user = ("Convert the following Python code to C++. Return only the C++ code, "
                "no explanation.\n\n```python\n" + text.strip() + "\n```")
    else:  # NL -> Python
        user = ("Generate Python code for the following description. Return only the "
                "Python code, no explanation.\n\n" + text.strip())
    messages = [{"role": "system", "content": "You are Qwen, a helpful coding assistant."},
                {"role": "user", "content": user}]
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def extract_code(text):
    text = text.strip()
    if "```" in text:
        parts = text.split("```")
        if len(parts) >= 2:
            block = parts[1]; lines = block.split("\n")
            if lines and lines[0].strip().lower() in ("python", "py", "cpp", "c++"):
                block = "\n".join(lines[1:])
            return block.strip()
    return text

@torch.no_grad()
def generate(task, text):
    if not text or not text.strip(): return "// enter some input first"
    model.set_adapter(task)                         # pick the right model
    inputs = tok(build_prompt(task, text), return_tensors="pt",
                  truncation=True, max_length=2048).to(DEVICE)
    out = model.generate(**inputs, max_new_tokens=512, do_sample=False,
                          pad_token_id=tok.eos_token_id)
    new = out[0][inputs["input_ids"].shape[1]:]
    return extract_code(tok.decode(new, skip_special_tokens=True))

In [ ]:
import gradio as gr
with gr.Blocks(title="Group-30 CodeGen") as demo:
    gr.Markdown("# Group-30 CodeGen\nPick a task, enter input, get generated code.")
    task = gr.Dropdown(TASKS, value=TASKS[0], label="Task")
    inp  = gr.Textbox(lines=8, label="Input",
                      placeholder="Describe the function, or paste Python code...")
    btn  = gr.Button("Generate", variant="primary")
    out  = gr.Code(label="Output", language="python")
    def _run(task, text):
        lang = "cpp" if task == "Python -> C++" else "python"
        return gr.update(value=generate(task, text), language=lang)
    btn.click(_run, [task, inp], out)
demo.launch(share=True)